# Mark a condition with phase

Goal: attach phase -1 to the values x<2 in a uniform two-bit register. A quantum condition acts coherently on amplitudes; Python `if` would not mean this operation. Here `control` guards a scalar phase. Measuring immediately cannot detect that phase, so we undo the initial Hadamards to make interference visible.

In [ ]:
import flagquantum as fq
from flagquantum import synth
import math
from flagquantum.synth import Output, QNum, allocate, control, hadamard_transform, phase, qfunc

@qfunc
def mark_and_interfere(x: Output[QNum]) -> None:
    allocate(2, x)
    hadamard_transform(x)
    control(x < 2, lambda: phase(1, math.pi))
    hadamard_transform(x)


In [ ]:
program = synth.synthesize(mark_and_interfere, max_qubits=8, memory_limit_bytes=1024**2)
result = fq.run(program)
outcomes = dict(program.probabilities(result, registers=("x",)))
print(outcomes)
print(program.report["resources"])


## Interpret the interference

The ideal output is x=2 with probability 1, up to floating-point roundoff. Before the final Hadamards the probabilities would still all be 1/4. Try changing the predicate to `x==1`: it produces a different interference pattern. The compiler implements the bounded predicate and its temporary cleanup. This example needs no manual comparator or phase-kickback ancilla.